In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download necessary NLTK data (run this once)
nltk.download('stopwords')
nltk.download('wordnet')

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text by converting to lowercase, removing URLs, emojis, and punctuation,
    removing stopwords, and applying stemming or lemmatization.
    """
    # Ensure dataset is a string
    data = str(dataset)

    # Convert the input to small/lower order.
    data = data.lower()

    # Remove URLs
    data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)

    # Remove emojis and all other unwanted characters (keep only alphanumeric and spaces)
    data = re.sub(r'[^\w\s]', '', data)

    # Create tokens.
    tokens = data.split()

    # Remove stopwords:
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Apply Lemmatization or Stemming
    if rule == "lemmatize":
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Load the Dataset
# Ensure the CSV file is in the correct path or uploaded to Colab
df = pd.read_csv("/content/drive/MyDrive/week8 ai ml/Copy of trum_tweet_sentiment_analysis.csv")

# Ensure there are no null values in the text column
df = df.dropna(subset=['text', 'Sentiment'])

# 2. Text Cleaning and Tokenization
# Apply the pipeline to the "text" column using pandas .apply()
print("Cleaning text data... this may take a moment.")
df['cleaned_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

# 3. Train-Test Split
# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['Sentiment'],
    test_size=0.2,
    random_state=42
)

# 4. TF-IDF Vectorization
# Convert the raw text into numerical features
vectorizer = TfidfVectorizer()

# Fit on training data and transform it
X_train_tfidf = vectorizer.fit_transform(X_train)
# Only transform the test data (do NOT fit to avoid data leakage)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Model Training and Evaluation
# Initialize and train the Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Print the classification report
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

Cleaning text data... this may take a moment.

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.95      0.97      0.96    248563
           1       0.93      0.90      0.91    121462

    accuracy                           0.95    370025
   macro avg       0.94      0.93      0.94    370025
weighted avg       0.94      0.95      0.94    370025

